<a href="https://colab.research.google.com/github/SunnyChoudhary850/FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

##The Data Contract (Lane 2: Refresh / Content Opportunity Scoring)

**What one row means:** one row = one content page, for one client, on one
calendar day (from `fact_content_daily_performance`). For modeling, I'll
aggregate this to one row per (client, content) pair per month.

**Tables used:** `dim_clients` (client metadata), `dim_content` (page
metadata), `fact_content_daily_performance` (daily metrics), filtered to
month = '2026-03'.

**Time window:** month = 2026-03 (a mid-panel month, not the sealed final
month 2026-06, which the warehouse's own warning says to treat as a test
month only).

**Target/proxy:** whether a page's performance is declining — comparing
this month's totals against the previous month's totals for the same page.

**One thing I deliberately exclude:** rows where `gsc_data_available` is
False — a page with no real Search Console data that day shouldn't be
treated as "zero performance," it should be excluded entirely, since zero
and missing mean different things here.

In [ ]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, gsc_impressions
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    ORDER BY client_hash_id, content_hash_id, report_date
    LIMIT 10
""").df()
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-01,0
1,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-02,0
2,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-03,0
3,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-04,0
4,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-05,0
5,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-06,0
6,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-07,0
7,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-08,0
8,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-09,0
9,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-10,0


This confirms the grain: one row per client + content + single calendar
day within March 2026. No duplicate (client, content, date) combinations.

In [ ]:
slice_info = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS earliest,
           MAX(report_date) AS latest
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
slice_info

,row_count,earliest,latest
0,9841378,2026-03-01,2026-03-31


####The Trap — Deliberate Leakage

First try: I added `impressions_next_month` as a feature, expecting the
score to jump toward perfect. It barely moved (0.65 → 0.67). Turned out my
"honest" features weren't even including `total_impressions_month`, so the
leaked column only had half the picture — it couldn't actually cheat yet.

I fixed that, but the leaked score still barely moved. Checked the label
balance first (59.6% declining, 40.4% not) — so 0.65 wasn't even much
better than just guessing the majority class. Turned out the real issue was
feature scaling: some features (raw impression counts) are way bigger
numbers than others (like days_with_activity, 0-31), and Logistic
Regression struggles to learn properly when features are on very different
scales. After scaling everything, the leak showed up clearly: honest score
stayed at 0.651, leaked score jumped to 0.879.

That jump is the leakage lesson from notebook 02, reproduced here on real
warehouse data. I removed `impressions_next_month` from my final feature
set and kept only the honest 0.651 score.

**One named limitation of my slice:** using only March 2026 as the feature
month captures one period only — content and search trends can be
seasonal, so patterns from this single month may not generalize to other
times of year.The Trap — Deliberate Leakage


In [ ]:
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [ ]:
features = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS total_impressions_month,
           AVG(gsc_avg_position) AS avg_position_month,
           COUNT(DISTINCT report_date) AS days_with_activity,
           SUM(gsc_clicks) AS total_clicks_month,
           SUM(ga4_sessions) AS total_sessions_month
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()
features.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,total_impressions_month,avg_position_month,days_with_activity,total_clicks_month,total_sessions_month
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.209549,31,7.0,1.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,2.987198,31,0.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.724039,31,6.0,3.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,7.244844,31,13.0,2.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,4.209227,31,1.0,2.0
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,9.445635,31,1.0,2.0
6,client_73cda7b4e4f265ea,content_1f380a642aed423b,96.0,6.014516,31,1.0,8.0
7,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,9.155335,31,1.0,0.0
8,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,5.258331,31,20.0,12.0
9,client_73cda7b4e4f265ea,content_20403327d8d9374c,3561.0,8.834415,31,10.0,22.0


##Five Features — available at the decision moment?

**total_impressions_month** — sum of past daily data within March, nothing from the future.

**avg_position_month** — average of past daily positions, historical only.

**days_with_activity** — count of past days with real GSC data, historical only.

**total_clicks_month** — sum of past daily clicks, historical only.

**total_sessions_month** — sum of past daily GA4 sessions, historical only.


All five are safe: computed only from March data, none reach into April or beyond.

In [ ]:
labels = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_next_month
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-04' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

merged = features.merge(labels, on=['client_hash_id','content_hash_id'], how='inner')
merged['declining'] = merged['impressions_next_month'] < merged['total_impressions_month']
merged = merged.dropna()

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

y = merged['declining']

# honest — includes total_impressions_month this time
X_honest = merged[['total_impressions_month','avg_position_month','days_with_activity','total_clicks_month','total_sessions_month']]
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
scaler = StandardScaler().fit(X_train)
model = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
print("Honest score:", model.score(scaler.transform(X_test), y_test))

# leaked — same features + the leaked column
X_leaked = merged[['total_impressions_month','avg_position_month','days_with_activity','total_clicks_month','total_sessions_month','impressions_next_month']]
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_leaked, y, test_size=0.3, random_state=42)
scaler2 = StandardScaler().fit(X_train2)
model2 = LogisticRegression(max_iter=1000).fit(scaler2.transform(X_train2), y_train2)
print("Leaked score (should jump way up):", model2.score(scaler2.transform(X_test2), y_test2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest score: 0.6507495022836398
Leaked score (should jump way up): 0.8788207050005855


## Self-Check

- Five plain-words contract answers given
- Three verification queries run with visible output (grain, row count/span, availability with IS TRUE)
- Five-feature frame built, each with an "available when?" line
- Deliberate leak shown (scaled version), then removed, honest 0.648 score kept
- One named limitation of the data slice